In [ ]:
!pip install --upgrade gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 25.4 MB/s eta 0:00:00


# **Proyecto 04: Sistema de Sentimiento en Reseñas de Usuario**

---
## **Parte 1: Carga del data set en la librería de Python**

En este proyecto, en lugar de cargar documentos de tipo '.csv' o de tipo '.txt', vamos a utilizar una librería de Python que ya tiene el dataset integrado. Para ello, importaremos la librería [**gensim**](https://radimrehurek.com/gensim/), y llamaremos a la función [**load_word2vec_format**](https://radimrehurek.com/gensim/models/word2vec.html). Con esta función cargaremos un modelo preconstruido de Word2Vec, propocionado por Google, que ha sido entrenado sobre billones de páginas de texto y que ha producido como salida un vector de dimensión 300 para todas las diferentes palabras. Una vez cargado el modelo, se puede ver el vector de 300 valores que representa una palabra en concreto, como por ejemplo <span style="color:red"> **cat**</span> o <span style="color:red"> **dog**</span>.

In [ ]:
import gensim.downloader as api

word2vec_model = api.load('word2vec-google-news-300')
vector_cat = word2vec_model['cat']
print(f"\nVector para la palabra 'cat' (Primeros 10 valores de 300):\n{vector_cat[:10]}")

[==================================================] 100.0% 1662.8/1662.8MB downloaded

Vector para la palabra 'cat' (Primeros 10 valores de 300):
[ 0.0123291   0.20410156 -0.28515625  0.21679688  0.11816406  0.08300781
  0.04980469 -0.00952148  0.22070312 -0.12597656]


Podemos incluso mirar el coeficiente de similaridad entre dos palabras distintas, utilizando la función [**similarity**](https://tedboy.github.io/nlps/generated/generated/gensim.models.Word2Vec.similarity.html).

In [ ]:
similitud_cat_dog = word2vec_model.similarity('cat', 'dog')
print(f"Similitud entre 'cat' y 'dog': {similitud_cat_dog}")

Similitud entre 'cat' y 'dog': 0.760945737361908


Para entrenar el modelo de Word2Vec, cargamos las clases <span style="color:blue"> **Doc2Vec**</span> y <span style="color:blue"> **TaggedDocument**</span>. Vamos a utilizar la primera porque queremos determinar un vector para cada documento, no necesariamente para cada palabra en el documento, ya que nuestros documentos son reseñas y nosotros queremos ver si esas reseñas son positivas o negativas, lo que significa comprobar si son similares a reseñas positivas o similares a reseñas negativas. <span style="color:blue"> **Doc2Vec**</span> lo proporciona la librería <span style="color:blue"> **gensim**</span> y también contiene la clase <span style="color:blue"> **TaggedDocument**</span>, que permite usar "estas son las palabras en el documento, y Doc2Vec es el modelo".

In [ ]:
from gensim.models.doc2vec import Doc2Vec, TaggedDocument

Creamos a continuación una función de utilidad que tomará una frase o un párrafo entero y le quitará las mayúsculas y eliminará las etiquetas HTML, apóstrofes, puntuación, espacios y espacios repetidos, y por último devolverá el párrafo separado por palabras (utilizad la función [**split**](https://www.w3schools.com/python/ref_string_split.asp) para esto último.

In [ ]:
import re

def extract_words(sentence):
    sentence = sentence.lower()
    sentence = re.sub(r'<[^>]+>', ' ', sentence)
    sentence = re.sub(r'[^\w\s]', ' ', sentence)
    sentence = re.sub(r'\s+', ' ', sentence)
    words = sentence.strip().split()
    return words

---

## **Parte 2: Carga de los datos**

Ahora vamos a cargar los datos sobre las reseñas de películas. En esta ocasión vamos a utilizar reseñas que no están catalogadas ni como positivas ni como negativas para entrenar el modelo; vamos a efectuar lo que se conoce como un unsupervised learning. Para cargar los datos, necesitamos hacer un bucle sobre el directorio donde se encuentran los archivos, así como otro bucle sobre los archivos de dicho directorio, tal y como se muestra a continuación

In [ ]:
!wget https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz
!tar -xf aclImdb_v1.tar.gz
!wget http://www.cs.cornell.edu/people/pabo/movie-review-data/review_polarity.tar.gz
!tar -xf review_polarity.tar.gz
!wget http://nlp.stanford.edu/~socherr/stanfordSentimentTreebank.zip
!unzip -o stanfordSentimentTreebank.zip

--2026-05-19 16:55:47--  https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz
Resolving ai.stanford.edu (ai.stanford.edu)... 171.64.68.10
Connecting to ai.stanford.edu (ai.stanford.edu)|171.64.68.10|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 84125825 (80M) [application/x-gzip]
Saving to: ‘aclImdb_v1.tar.gz’

aclImdb_v1.tar.gz   100%[===================>]  80.23M  17.7MB/s    in 9.6s    

2026-05-19 16:55:57 (8.40 MB/s) - ‘aclImdb_v1.tar.gz’ saved [84125825/84125825]

--2026-05-19 16:56:10--  http://www.cs.cornell.edu/people/pabo/movie-review-data/review_polarity.tar.gz
Resolving www.cs.cornell.edu (www.cs.cornell.edu)... 132.236.207.53
Connecting to www.cs.cornell.edu (www.cs.cornell.edu)|132.236.207.53|:80... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://www.cs.cornell.edu/people/pabo/movie-review-data/review_polarity.tar.gz [following]
--2026-05-19 16:56:11--  https://www.cs.cornell.edu/people/pabo

In [ ]:
import os
import random
from gensim.models.doc2vec import TaggedDocument
from sklearn.model_selection import train_test_split

train_sentences = []
test_sentences = []


# 1. Base de datos IMDB
imdb_temp = []
for dirname in ["train/pos", "train/neg", "train/unsup", "test/pos", "test/neg"]:
    for fname in sorted(os.listdir("aclImdb/" + dirname)):
        if fname[-4:] == '.txt':
            with open("aclImdb/" + dirname + "/" + fname, encoding='UTF-8') as f:
                sent = f.read()
                words = extract_words(sent)
                imdb_temp.append(TaggedDocument(words, [dirname + "/" + fname]))

imdb_train, imdb_test = train_test_split(imdb_temp, test_size=0.20, random_state=42)
train_sentences.extend(imdb_train)
test_sentences.extend(imdb_test)


# 2. Base de datos review_polarity
cornell_temp = []
for dirname in ["txt_sentoken/pos", "txt_sentoken/neg"]:
    for fname in sorted(os.listdir(dirname)):
        if fname[-4:] == '.txt':
            with open(dirname + "/" + fname, encoding='UTF-8') as f:
                for i, sent in enumerate(f):
                    words = extract_words(sent)
                    cornell_temp.append(TaggedDocument(words, ["%s/%s-%d" % (dirname, fname, i)]))

cornell_train, cornell_test = train_test_split(cornell_temp, test_size=0.20, random_state=42)
train_sentences.extend(cornell_train)
test_sentences.extend(cornell_test)

# 3. Base de datos Rotten Tomatoes
rt_temp = []
with open("stanfordSentimentTreebank/original_rt_snippets.txt", encoding='UTF-8') as f:
    for i, line in enumerate(f):
        words = extract_words(line)
        rt_temp.append(TaggedDocument(words, ["rt-%d" % i]))

rt_train, rt_test = train_test_split(rt_temp, test_size=0.20, random_state=42)
train_sentences.extend(rt_train)
test_sentences.extend(rt_test)

# Comprobación final
print(f"Total de documentos para ENTRENAR (80%): {len(train_sentences)}")
print(f"Total de documentos para TESTEAR (20%): {len(test_sentences)}")

Total de documentos para ENTRENAR (80%): 140260
Total de documentos para TESTEAR (20%): 35065


---

## **Parte 3: Análisis del model (entrenamiento)**

El entrenamiento sin supervisión lo realizamos primero cambiando de orden los datos de entrenamiento (con la función <span style="color:blue"> **shuffle**</span>) y entrenamos llamando al modelo <span style="color:blue"> **Doc2Vec**</span> (utilizad los parámetros dm=0, hs=1 y size=50). Puede tardar entre 5 y 10 minutos en efectuar este entrenamiento.

In [ ]:
import random
from gensim.models.doc2vec import Doc2Vec

random.shuffle(train_sentences)
model = Doc2Vec(train_sentences, dm=0, hs=1, vector_size=50, epochs=10)

Una vez terminado el entrenamiento, eliminamos los datos de entrenamiento con la función <span style="color:blue"> **delete_temporary_training_data**</span> y guardamos el modelo en un archivo ".d2v".

In [ ]:
model.save("modelo_sentimientos.d2v")

Una vez el modelo ha sido entrenado, podemos inferir un vector. Probad a mandar una reseña a la función infer_vector, por ejemplo "This place is not worth your time, let alone Vegas.". Podéis probar con otras como "Service sucks" o "Highly recommended". Si queréis comparar entre dos reseñas, podéis llamar a similarity, aunque
podemos probar con [**cosine_similarity**](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.cosine_similarity.html)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np


frase1 = "This place is not worth your time, let alone Vegas."
frase2 = "Service sucks"
frase3 = "Highly recommended"

vec1 = model.infer_vector(extract_words(frase1))
vec2 = model.infer_vector(extract_words(frase2))
vec3 = model.infer_vector(extract_words(frase3))

sim_negativas = cosine_similarity(vec1.reshape(1, -1), vec2.reshape(1, -1))[0][0]
sim_opuestas = cosine_similarity(vec1.reshape(1, -1), vec3.reshape(1, -1))[0][0]

print(f"Similitud entre '{frase1}' y '{frase2}': {sim_negativas:.4f}")
print(f"Similitud entre '{frase1}' y '{frase3}': {sim_opuestas:.4f}")

Similitud entre 'This place is not worth your time, let alone Vegas.' y 'Service sucks': 0.4515
Similitud entre 'This place is not worth your time, let alone Vegas.' y 'Highly recommended': 0.3002


El modelo ha aprendido como se usan las palabras juntas en la misma reseña y que un grupo de palabras pueden ir juntas con un significado, y otro grupo de palabras pueden ir juntas con otro resultado.

A continuación cargamos el dataset para predecir (aquellas carpetas que sí que tienen etiquetas)

In [ ]:
import os

sentences = []
sentiments = []

# 1. Dataset aclImdb (IMDB)
for dirname in ["train/pos", "train/neg", "test/pos", "test/neg"]:
    base_path = "aclImdb/" + dirname
    if os.path.exists(base_path):
        for fname in sorted(os.listdir(base_path)):
            if fname.endswith('.txt'):
                with open(os.path.join(base_path, fname), encoding='UTF-8') as f:
                    words = extract_words(f.read())
                    sentences.append(words)
                    sentiments.append(1 if "/pos" in dirname else 0)

# 2. Dataset txt_sentoken (Cornell Movie Review Data)
for dirname in ["txt_sentoken/pos", "txt_sentoken/neg"]:
    if os.path.exists(dirname):
        for fname in sorted(os.listdir(dirname)):
            if fname.endswith('.txt'):
                with open(os.path.join(dirname, fname), encoding='UTF-8') as f:
                    # Este dataset suele tener una frase por línea o el documento completo
                    full_text = f.read()
                    words = extract_words(full_text)
                    if words:
                        sentences.append(words)
                        sentiments.append(1 if "/pos" in dirname else 0)

# 3. Dataset stanfordSentimentTreebank (Rotten Tomatoes)
snippets_path = "stanfordSentimentTreebank/original_rt_snippets.txt"
if os.path.exists(snippets_path):
    with open(snippets_path, encoding='UTF-8') as f:
        for line in f:
            words = extract_words(line)
            if words:
                sentences.append(words)
                sentiments.append(1)

print(f"Total reseñas cargadas: {len(sentences)}")

Total reseñas cargadas: 62605


Para clasificar este modelo entrenado que hemos obtenido, vamos a utilizar un
clasificador de k-vecinos cercanos, utilizando la clase [*+KNeighborsClassifier**](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html) y un [RandomForestClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html), así como la función [cross_val_score](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_score.html) para evaluar y verificar los resultados.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
import numpy as np

print("Convirtiendo reseñas en vectores... esto puede tardar un momento.")
sentvecs = [model.infer_vector(s) for s in sentences]

X = np.array(sentvecs)
y = np.array(sentiments)

print("--- Evaluación de Modelos ---")

# 1. Clasificador K-Vecinos Cercanos (KNN)
knn_clf = KNeighborsClassifier(n_neighbors=5)
knn_scores = cross_val_score(knn_clf, X, y, cv=5, scoring='accuracy')
print(f"KNN -> Precisión Media: {knn_scores.mean():.4f} (+/- {knn_scores.std() * 2:.4f})")

# 2. Clasificador Random Forest
rf_clf = RandomForestClassifier(n_estimators=100, random_state=42)
rf_scores = cross_val_score(rf_clf, X, y, cv=5, scoring='accuracy')
print(f"Random Forest -> Precisión Media: {rf_scores.mean():.4f} (+/- {rf_scores.std() * 2:.4f})")

Convirtiendo reseñas en vectores... esto puede tardar un momento.
--- Evaluación de Modelos ---
KNN -> Precisión Media: 0.7725 (+/- 0.0361)
Random Forest -> Precisión Media: 0.8196 (+/- 0.1109)


In [ ]:
import joblib

print("Entrenando modelos finales con todos los datos...")
knn_clf.fit(X, y)
rf_clf.fit(X, y)

print("Guardando los modelos...")
joblib.dump(knn_clf, 'modelo_knn_final.pkl')
joblib.dump(rf_clf, 'modelo_rf_final.pkl')

print("¡Modelos guardados con éxito!")

Entrenando modelos finales con todos los datos...
Guardando los modelos...
¡Modelos guardados con éxito!


In [ ]:
import numpy as np

def probar_resena(texto):
    print(f"\n--- Analizando: '{texto[:100]}...' ---")
    palabras_limpias = extract_words(texto)
    vector = model.infer_vector(palabras_limpias)
    vector_modelo = vector.reshape(1, -1)

    prediccion_knn = knn_clf.predict(vector_modelo)[0]
    prediccion_rf = rf_clf.predict(vector_modelo)[0]

    probabilidad_rf = rf_clf.predict_proba(vector_modelo)[0]

    sentimiento_knn = "Positivo +" if prediccion_knn == 1 else "Negativo -"
    sentimiento_rf = "Positivo +" if prediccion_rf == 1 else "Negativo -"

    print(f"KNN opina que es: {sentimiento_knn}")
    print(f"Random Forest opina que es: {sentimiento_rf}")
    print(f"   ↳ Seguridad del Random Forest: {probabilidad_rf[1]*100:.1f}% Positivo / {probabilidad_rf[0]*100:.1f}% Negativo")


print("\n" + "="*50)
print("ANALIZANDO LISTA TEST_SENTENCES")
print("="*50)

for doc in test_sentences[:5]:
    texto_original = " ".join(doc.words)
    probar_resena(texto_original)